# Projection and Rectification
This example script projects raw depth maps to the target (left) event camera, then rectifies the projected depth maps and the left and right event data. First, load the required libraries.

In [ ]:
import cv2
import h5py
import hdf5plugin
import os
import numpy as np
from glob import glob
from event_reader.eventslicer import EventSlicer
from matplotlib import pyplot as plt
from calibration_loader.EventKitchen_Calibration import Calibration_Loader

Then enter the paths to the left and right event files.

In [ ]:
# load events
leftevent_file = "" # path to the dataset/LeftEvent/LeftEvent.hdf5 
rightevent_file = "" # path to the dataset/RightEvent/RightEvent.hdf5
leftevent_loader = EventSlicer(h5py.File(leftevent_file, 'r'))
rightevent_loader = EventSlicer(h5py.File(rightevent_file, 'r'))

<span style="color: red;">**Note:**</span> We use a subset of the dataset to evaluate the stereo depth estimation baselines reported in the paper. This subset excludes sequences with invalid time periods listed in [`read_event_file.ipynb`](read_event_file.ipynb) and [`defect_data_stats.csv`](defect_data_stats.csv). The subset is listed below:

**Training set** [`train_depth_map.csv`](train_depth_map.csv)

| Session 01 | Session 02 | Session 05 | Session 06 | Session 07 | Session 08 | Session 09 | Session 11 | Session 12 | Session 13 |
| :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| vege_salad | cereal_bowl | fry_bacon | sandwich | fry_egg | lemon_water | sandwich | cut_bread | fry_pepper | fruit_salad |
| cereal_bowl | cut_bread | fry_egg | tea_1 | cut_cake | coffee | wash_dish | sandwich | fry_bacon | vege_salad |
| - | - | - | cut_bread | - | - | tea | - | - | - |

**Test set** [`test_depth_map.csv`](test_depth_map.csv)

| Session 03 | Session 04 | Session 10 | Session 14 |
| :---: | :---: | :---: | :---: |
| cut_bread | coffee | fry_pepper | wash_dish |
| tea_2 | vege_salad | fry_egg | cereal_bowl |
| coffee | - | - | cut_bread |

Then enter the path to the extracted depth maps. All depth maps (.tiff) under this path are loaded into ***depth_maps***. We suggest adding code to check whether the depth maps and event file match (the same activity in the same recording session).

In [ ]:
# load depth maps
depth_map_path = "" # the path to the saved depth maps
depth_maps = glob(os.path.join(depth_map_path, '*.tiff'))

####
## code to check the whether the annotation and data are matched
####

Then enter the path to the calibration parameters.

In [ ]:
# load calibration 
calibration_path = "" # the path to the saved calibration results, Calibration/calibration_results_d435/ or Calibration/calibration_results_d455
calibrator = Calibration_Loader(calibration_path)

Next, we perform projection and rectification. For the reported baselines, we first accumulate the raw events into event representations, then rectify the depth map and event representations. Therefore, a function for creating event representations is needed.

<span style="color: red;">**Note:**</span> You can also rectify the raw events using the provided rectification map.

In [ ]:
def event_representation(event_slice):
    ####
    ## code to create the event representation
    ####

    return representation

Define a time window for creating the event representation and aligning it with the depth map, for example, 1/30 s (approximately 33.3 ms).

In [ ]:
### start to rectify data
deltaT = 1/30 # the window length of the event slice to align the depth map

In the following code, the for loop first loads a depth map, projects it into the field of view of the left event camera, and rectifies the projected depth map and the left and right event representations.

In [ ]:
for m in depth_maps:
    ts = float(m.split('/')[-1][:-4]) # the timestamp of the depth map
    event_start_ts, event_end_ts = (ts - deltaT) * 1e6, ts * 1e6 # the timestamp of the aligned event slice
    ### align the first and final depth map
    if event_end_ts < leftevent_loader.get_start_time_us():
        continue
    if event_start_ts < leftevent_loader.get_start_time_us():
        event_start_ts = leftevent_loader.get_start_time_us()
    if event_end_ts > leftevent_loader.get_final_time_us():
        event_end_ts = leftevent_loader.get_final_time_us()
    if event_start_ts > leftevent_loader.get_final_time_us():
        break
    
    depth = cv2.imread(m, -1).astype(np.uint16) # read depth
    leftevent_slice = leftevent_loader.get_events(event_start_ts, event_end_ts) # load the aligned left event
    rightevent_slice = rightevent_loader.get_events(event_start_ts, event_end_ts) # load the aligned right event

    # convert raw events to representations
    leftevent_representation = event_representation(leftevent_slice)
    rightevent_representation = event_representation(rightevent_slice)
    
    ## project depth to the fov of left event camera, unit is in millimeter
    # undistort depth
    map1, map2 = cv2.initUndistortRectifyMap(
        calibrator.DRGB_intrinsic_matrix, 
        calibrator.DRGB_distortion_matrix, 
        R=np.eye(3), 
        newCameraMatrix=calibrator.DRGB_intrinsic_matrix, 
        size=(1280, 720), 
        m1type=cv2.CV_32FC1
    )
    depth_undist = cv2.remap(
        depth, map1, map2,
        interpolation=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0
    )
    # project depth
    proj_depth2left = calibrator.project_depth_to_event(
        depth0_mm=depth_undist,
        K0=calibrator.DRGB_intrinsic_matrix,
        K1=calibrator.LeftEvent_intrinsic_matrix,
        R_0to1=calibrator.rotation_matrix_DRGB_LeftEvent,
        T_0to1_mm=calibrator.translation_matrix_DRGB_LeftEvent  
    )

    ## rectify projected depth, left event, and right event
    # rectify projected depth
    rectified_LeftDepth = cv2.remap(proj_depth2left,
                                    calibrator.stereoMapLeftEvent_X,
                                    calibrator.stereoMapLeftEvent_Y,
                                    cv2.INTER_LINEAR)
    # rectify left event
    rectified_LeftEvent = cv2.remap(leftevent_representation,
                                    calibrator.stereoMapLeftEvent_X,
                                    calibrator.stereoMapLeftEvent_Y,
                                    cv2.INTER_LINEAR)

    # rectify right event
    rectified_RightEvent = cv2.remap(rightevent_representation,
                                     calibrator.stereoMapRightEvent_X,
                                     calibrator.stereoMapRightEvent_Y,
                                     cv2.INTER_LINEAR)
    
    ####
    ## your code for data processing
    ####